In [40]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch.distributions import Independent, Uniform, Normal, MultivariateNormal
from sbi import analysis, utils
from sbi.inference import NPE, simulate_for_sbi
from sbi.utils.user_input_checks import (
    check_sbi_inputs,
    process_prior,
    process_simulator,
)
import math
import sympy as sp

seed = 0
torch.manual_seed(seed);


C = 299792.458 # km/s

In [41]:
def calc_hubble_distance(hubble, matter, curv, z):
    matter_term = matter * (1 + z)**3
    curvature_term = curv * (1 + z)**2
    lambda_term = 1 - matter - curv
    hubble_term = hubble * (1 + z)
    return torch.max(hubble * torch.sqrt(matter_term + curvature_term + lambda_term), torch.tensor(0.0001))

def calc_luminosity_distance(hubble, matter, curv, z):
    hubble_distance = calc_hubble_distance(hubble, matter, curv, z)
    z_values = torch.linspace(0, z, 1000)
    hubble_distance_values = calc_hubble_distance(hubble, matter, curv, z_values)
    return C * torch.trapz(1 / hubble_distance_values, z_values)

def calc_apparent_magnitude(hubble, matter, curv, z):
    luminosity_distance = calc_luminosity_distance(hubble, matter, curv, z)
    return 5 * torch.log10(luminosity_distance) + 25

def load_real_data():
    """Load and prepare the supernova data"""
    import pandas as pd
    df = pd.read_csv('/Users/yhra/Documents/Master/Semester_3/BATIP/Supernova_project/Data/Pantheon+SH0ES.dat', sep='\s+')
    
    # Filter data
    mask = df['zCMB'] > 0.001
    z_obs = df['zCMB'][mask].values
    mu_obs = df['MU_SH0ES'][mask].values
    mu_err = df['MU_SH0ES_ERR_DIAG'][mask].values
    
    # Convert to tensors
    z_obs = torch.as_tensor(z_obs, dtype=torch.float32).clone().detach()
    mu_obs = torch.as_tensor(mu_obs, dtype=torch.float32).clone().detach()
    mu_err = torch.as_tensor(mu_err, dtype=torch.float32).clone().detach()

    # Create covariance matrix
    cov_data = np.loadtxt('/Users/yhra/Documents/Master/Semester_3/BATIP/Supernova_project/Data/Pantheon+SH0ES_STATONLY.cov')
    cov_matrix = _create_cov_mat(cov_data)
    
    return z_obs, mu_obs, mu_err, cov_matrix

def _create_cov_mat(cov_data):
    n = int(cov_data[0])
    cov_matrix = cov_data[1:].reshape(n, n)
    
    # Make symmetric if needed
    if not np.allclose(cov_matrix, cov_matrix.T):
        cov_matrix = (cov_matrix + cov_matrix.T) / 2
    return cov_matrix

# Load real data
z_obs, mu_obs, mu_err, cov_matrix = load_real_data()


In [58]:
class SBIConfig:
    # Add random seed to config
    RANDOM_SEED = 42
    
    # SBI settings
    NUM_SIMULATIONS = 10000
    NUM_POSTERIOR_SAMPLES = 1000000  # Increased for better posterior visualization
    BATCH_SIZE = 256    
    # Physics constants
    C = 299792.458  # Speed of light in km/s
    
    # Data filtering
    MIN_REDSHIFT = 0.001
    
    # Paths
    DATA_PATH = '/Users/yhra/Documents/Master/Semester_3/BATIP/Supernova_project/Data/'
    PLOT_PATH = '/Users/yhra/Documents/Master/Semester_3/BATIP/Supernova_project/Plots/'
    
    
    EPSILON = 1e-5
    DEVICE = torch.device("cpu")
    
    # Parameter priors (mean, std)
    PARAM_PRIORS = {
        'flat': {
            'H0': (70.0, 0.5),  # Tighter constraint around 71
            'Om': (0.2, 0.03)   # Tighter constraint around 0.3
        },
        'curved': {
            'H0': (70.0, 0.5),
            'Om': (0.2, 0.03),
            'Ok': (0.0, 0.001)  # Much tighter constraint around 0
        }
    }

class CosmologicalSimulatorInference:
    def __init__(self, z_obs, mu_obs, cov_matrix = None, model_type="flat"):
        """Initialize the simulator with observed redshift values"""
        # set_seed(SBIConfig.RANDOM_SEED)  # Set seed in constructor
        self.z = torch.as_tensor(z_obs, dtype=torch.float32).clone().detach()
        self.mu = torch.as_tensor(mu_obs, dtype=torch.float32).clone().detach()
        self.model_type = model_type
        self.cov_matrix = cov_matrix
        self.prior = None
        self.setup_prior()
        self.Ok_sym, self.chi_sym = sp.symbols('Ok chi')
        self.setup_taylor_series(order=6)
        
    def setup_prior(self):
        """Setup the prior distribution for parameters"""
        if self.model_type == "flat":
            self.prior = Independent(
                Normal(
                    loc=torch.tensor([SBIConfig.PARAM_PRIORS['flat']['H0'][0],
                                    SBIConfig.PARAM_PRIORS['flat']['Om'][0]]),
                    scale=torch.tensor([SBIConfig.PARAM_PRIORS['flat']['H0'][1],
                                    SBIConfig.PARAM_PRIORS['flat']['Om'][1]])
                ),
                1
            )
        else:
            self.prior = Independent(
                Normal(
                    loc=torch.tensor([SBIConfig.PARAM_PRIORS['curved']['H0'][0],
                                    SBIConfig.PARAM_PRIORS['curved']['Om'][0],
                                    SBIConfig.PARAM_PRIORS['curved']['Ok'][0]]),
                    scale=torch.tensor([SBIConfig.PARAM_PRIORS['curved']['H0'][1],
                                    SBIConfig.PARAM_PRIORS['curved']['Om'][1],
                                    SBIConfig.PARAM_PRIORS['curved']['Ok'][1]])
                ),
                1
            )

    def setup_taylor_series(self, order=16):
        """Setup the Taylor series for the curvature correction"""
        f_positive = (1 / sp.sqrt(self.Ok_sym)) * sp.sinh(sp.sqrt(self.Ok_sym) * self.chi_sym)
        f_negative = (1 / sp.sqrt(-self.Ok_sym)) * sp.sin(sp.sqrt(-self.Ok_sym) * self.chi_sym)
        if self.model_type == "curved":
            self._taylor_series_negative = sp.series(f_positive, self.Ok_sym, 0, order).removeO()
            self._taylor_series_positive = sp.series(f_negative, self.Ok_sym, 0, order).removeO()
        
    def hubble_z(self, H0, Om, Ok=None):
        """Hubble parameter at redshift z"""
        # Reshape parameters to allow broadcasting with z
        H0 = H0.reshape(-1, 1)  # Shape: (batch_size, 1)
        Om = Om.reshape(-1, 1)  # Shape: (batch_size, 1)
        
        matter_term = Om * (1 + self.z)**3
        if self.model_type == "flat":
            lambda_term = (1 - Om)
            return H0 * torch.sqrt(matter_term + lambda_term)
        else:
            Ok = Ok.reshape(-1, 1)  # Shape: (batch_size, 1)
            lambda_term = (1 - Om - Ok)
            curvature_term = Ok * (1 + self.z)**2
            return H0 * torch.sqrt(matter_term + curvature_term + lambda_term)
    
    def luminosity_distance(self, H0, Om, Ok=None):
        """
        Compute luminosity distance with improved precision.
        Returns tensor of shape (batch_size, n_redshifts)
        """
        # Parameter validation
        if torch.any(Om < 0) or torch.any(Om > 1):
            return torch.full_like(torch.zeros((len(H0), len(self.z))), float('inf'))
        if self.model_type == "curved":
            if torch.any(torch.abs(Ok) > 1) or torch.any((Om + Ok) > 1):
                return torch.full_like(torch.zeros((len(H0), len(self.z))), float('inf'))

        # Create fine integration grid with more points for higher precision
        z_max = torch.max(self.z)
        n_points = 2000  # Increased from 1000 for better precision
        z_grid = torch.linspace(SBIConfig.MIN_REDSHIFT, z_max, n_points)

        # Reshape parameters for broadcasting
        H0_exp = H0.reshape(-1, 1)
        Om_exp = Om.reshape(-1, 1)
        
        # Calculate Hubble parameter
        matter_term = Om_exp * (1 + z_grid)**3
        if self.model_type == "flat":
            Hz = H0_exp * torch.sqrt(matter_term + (1 - Om_exp))
        else:
            Ok_exp = Ok.reshape(-1, 1)
            Hz = H0_exp * torch.sqrt(
                matter_term + 
                Ok_exp * (1 + z_grid)**2 + 
                (1 - Om_exp - Ok_exp)
            )

        # Compute integrand
        integrand = SBIConfig.C / Hz

        # Initialize output tensor
        d_L = torch.zeros((len(H0), len(self.z)))

        # Compute luminosity distance for each redshift
        for i, z in enumerate(self.z):
            z_val = float(z)
            if z_val >= SBIConfig.MIN_REDSHIFT:
                mask = z_grid <= z_val
                chi = torch.trapz(integrand[:, mask], z_grid[mask], dim=1) # dim=1 is the dimension of the batch
                # chi dim = (batch_size, 1)
                if self.model_type == "curved":
                    chi = self.apply_curvature_correction(chi, Ok.reshape(-1))
                
                d_L[:, i] = (1 + z_val) * chi

        return d_L
    
    def distance_modulus(self, H0, Om, Ok=None):
        """Compute distance modulus"""
        d_L = self.luminosity_distance(H0, Om, Ok)
        return 5 * torch.log10(d_L + SBIConfig.EPSILON) + 25
    
    def simulate(self, params, cov_matrix=None):
        """Simulate distance moduli for given parameters"""
        if params.ndim == 1:
            params = params.unsqueeze(0)
        
        if self.model_type == "flat":
            H0, Om = params[:, 0], params[:, 1]
            mu = self.distance_modulus(H0, Om)
        else:
            H0, Om, Ok = params[:, 0], params[:, 1], params[:, 2]
            mu = self.distance_modulus(H0, Om, Ok)
        
        if cov_matrix is not None:
            # Create error distribution for each simulation
            batch_size = len(params)
            errors = torch.zeros_like(mu)
            
            # Sample errors for each simulation independently
            for i in range(batch_size):
                error_dist = MultivariateNormal(
                    loc=torch.zeros_like(self.z),
                    covariance_matrix=cov_matrix
                )
                errors[i] = error_dist.sample()
            
            # Add errors to simulated distance moduli
            mu = mu + errors
        
        # Return both z and mu for each simulation
        return torch.stack([self.z.expand(len(params), -1), mu], dim=-1)
    
    def apply_curvature_correction(self, chi, Ok):
        """
        Apply curvature correction to an array of chi using SymPy's Taylor series expansion. 
        """
        print(chi)
        print(Ok)
        return chi
         


simulator = CosmologicalSimulatorInference(
                                    z_obs,
                                    mu_obs,
                                    cov_matrix,
                                    model_type="curved"
)

# draw 10000 samples from the prior
samples = simulator.prior.sample((100,))
# simulate the data
simulated_data = simulator.simulate(samples)[:,:,1]










tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.])
tensor([ 4.6271e-05,  1.2728e-03,  1.4144e-04,  2.7051e-04, -1.7019e-03,
         1.2853e-03,  4.7358e-04,  2.1526e-04, -5.7191e-04, -7.8843e-04,
         1.5694e-04,  1.8673e-03,  6.5246e-04, -5.0748e-04, -6.3197e-04,
         1.1693e-03, -7.0396e-04, -3.1151e-04,  1.6469e-03,  2.8391e-04,
        -4.5526e-05,  4.6309e-04, -5.7806e-04,  1.5246e-04, -1.3912e-03,
        -2.6278e-04, -1.0045e-03,  5.1047e-04,  5.4074e-04, -1.1917e-03,
         1.0590e-03,  7.5568e-05,  1.3788e-03, -1.0695e-03,  3.3560e-04,
         5.0104e-04,  3.2798e-04,  8.3360e-05,  

In [39]:
import sympy as sp

def taylor_series_sinh_sqrt(x_order=16):
    # Define symbols
    x, a = sp.symbols('x a')
    
    # Define the function
    f = (1 / sp.sqrt(x)) * sp.sinh(sp.sqrt(a) * x)
    
    # Compute the Taylor series expansion
    taylor_series = sp.series(f, x, 0, x_order + 1).removeO()
    
    return taylor_series

# Example usage
series = taylor_series_sinh_sqrt(16)
print(series)

# evaluate the series at x = 0.1 and a = 0.1
series.subs({'x': 0.1, 'a': 0.1}).evalf()



a**(17/2)*x**(33/2)/355687428096000 + a**(15/2)*x**(29/2)/1307674368000 + a**(13/2)*x**(25/2)/6227020800 + a**(11/2)*x**(21/2)/39916800 + a**(9/2)*x**(17/2)/362880 + a**(7/2)*x**(13/2)/5040 + a**(5/2)*x**(9/2)/120 + a**(3/2)*x**(5/2)/6 + sqrt(a)*sqrt(x)


0.100016667500020